In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

In [14]:
%cd /content/drive/MyDrive/repositories/aq_edge/colab
%run mlflow_colab.py

/content/drive/MyDrive/repositories/aq_edge/colab
MLflow server ready: http://127.0.0.1:5000


In [15]:
%cd /content/drive/MyDrive/repositories/aq_edge

import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

!python hyperparameter_optuna.py --optuna.n_trials 10 --device.use_cuda true

/content/drive/MyDrive/repositories/aq_edge/colab
[2026-10-20 19:11:59] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261020_191159.log
[2026-10-20 19:11:59] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261020_191159.log
[2026-10-20 19:12:01] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261020_191201.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261020_191201.log
[2026-10-20 19:12:01] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-20 19:12:01] INFO: Device: cuda
INFO:__main__:Device: cuda
[2026-10-20 19:12:01] INFO: Station: UTEC
INFO:__main__:Station: UTEC
[2026-10-20 19:12:01] INFO: Features: ['CO2', 'TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['CO2', 'TEM', 'HUM', 'VOC']
[2026-10-20 19:12:01] INFO: Target: CO2
INFO:__mai

In [17]:
import re

# Let's inspect how the model input_size and the dataset are configured in hyperparameter_optuna.py
optuna_file_path = "/content/drive/MyDrive/repositories/aq_edge/hyperparameter_optuna.py"

with open(optuna_file_path, "r") as f:
    lines = f.readlines()

# Print lines around the training preparation or objective definition
print("=== Looking for model initialization and train_trial inside hyperparameter_optuna.py ===")
for i, line in enumerate(lines):
    if "def train_trial" in line or "model = " in line or "input_size" in line:
        start = max(0, i - 5)
        end = min(len(lines), i + 10)
        print(f"\n--- Lines {start+1} to {end} ---")
        print("".join(lines[start:end]))

=== Looking for model initialization and train_trial inside hyperparameter_optuna.py ===

--- Lines 128 to 142 ---

# ============================================================
# Model initialization
# ============================================================

def build_model(params, input_size, output_size, device):
    model_classes = {
        "BaseLSTM": BaseLSTM,
        #"AttentionLSTM": AttentionLSTM,
    }

    model_cls = model_classes[params["model_type"]]

    model = model_cls(
        input_size=input_size,


--- Lines 136 to 150 ---
        #"AttentionLSTM": AttentionLSTM,
    }

    model_cls = model_classes[params["model_type"]]

    model = model_cls(
        input_size=input_size,
        output_size=output_size,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
    )

    return model.to(device)



--- Lines 137 to 151 ---
    }

    model_cls = model_classes[params["model_type"]]

    model = 

In [18]:
# Let's find how data and dataloaders are prepared in hyperparameter_optuna.py
with open(optuna_file_path, "r") as f:
    content = f.read()

# Let's search for how 'prepared_data' is created or where datasets are instantiated
lines = content.splitlines()
print("=== Inspecting dataset preparation/loading lines ===")
for i, line in enumerate(lines):
    if "prepare_data" in line or "dataset" in line.lower() or "DataLoader" in line or "features" in line:
        if any(keyword in line for keyword in ["def ", "=", "("]):
            start = max(0, i - 4)
            end = min(len(lines), i + 8)
            print(f"\n--- Lines {start+1} to {end} ---")
            print("\n".join(lines[start:end]))

=== Inspecting dataset preparation/loading lines ===

--- Lines 66 to 77 ---
    output_sequence_length,
    scaler_name,
):
    """
    Reuse the existing prepare_data() function.

    IMPORTANT:
    prepare_data() must support selecting the scaler. This helper
    detects common parameter names. If your function uses another
    name, add it to scaler_aliases below.

    The scaler must be fitted on training data only.

--- Lines 69 to 80 ---
    """
    Reuse the existing prepare_data() function.

    IMPORTANT:
    prepare_data() must support selecting the scaler. This helper
    detects common parameter names. If your function uses another
    name, add it to scaler_aliases below.

    The scaler must be fitted on training data only.
    """
    signature = inspect.signature(prepare_data)
    supported = signature.parameters

--- Lines 75 to 86 ---
    name, add it to scaler_aliases below.

    The scaler must be fitted on training data only.
    """
    signature = inspect.signat

In [19]:
import sys
sys.path.append("/content/drive/MyDrive/repositories/aq_edge")

# Let's inspect the actual shape of training batches to see why we have 5 features
import pandas as pd
import numpy as np
from aq_edge.utils.config import ConfigHandler

config_path = "/content/drive/MyDrive/repositories/aq_edge/config/config.yaml" # assuming default path
# Let's find config file path in hyperparameter_optuna.py content
for line in content.splitlines():
    if "config" in line.lower() and ".yaml" in line.lower():
        print("Config reference found:", line)

# Let's import the data preparation function from the optuna file directly if possible
# or manually load some data and call prepare_trial_data to check the batch shapes.

In [21]:
import os
from pathlib import Path

repo_root = Path("/content/drive/MyDrive/repositories/aq_edge")
print("Searching for YAML configuration files...")
for path in repo_root.rglob("*.yaml"):
    print(path)
for path in repo_root.rglob("*.yml"):
    print(path)

Searching for YAML configuration files...
/content/drive/MyDrive/repositories/aq_edge/config.yaml


In [ ]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

import optuna

output_dir = Path("optuna_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save all trials
study.trials_dataframe().to_csv(
    output_dir / "trials.csv",
    index=False,
)

# Save the best trial
if study.best_trial is not None:
    result = {
        "study_name": study.study_name,
        "best_trial": study.best_trial.number,
        "objective": "validation_r2",
        "best_value": study.best_value,
        "params": study.best_trial.params,
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }

    with (output_dir / "best_params.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_dir.resolve()}")